- assistants api => responses api
    - https://platform.openai.com/docs/guides/migrate-to-responses
    - Chat Completions API  for Chat AI, Responses API  for  agentic AI
        - Chat Completions API 则是一个完全无状态（Stateless） 的 API。每次请求都是独立的，不依赖于之前的任何会话信息，每次请求都包含了完整的上下文（messages），服务端不保存任何对话状态。
        -  Agentic AI 应用场景: 多轮推理、内置工具调用、远程 MCP 调用
    - The Responses API is our new API primitive, an evolution of Chat Completions which brings added simplicity and powerful agentic primitives to your integrations.
        - 更强的原生工具支持：除了 Function Calling，还内置了网页搜索（Web Search）、代码解释器（Code Interpreter）、Computer Use 等实用、进阶的 Agent 工具。
        - 显式的流式思维链（Reasoning Summary） ：模型在输出最终答案前能流式输出它的 “思考过程”，我们终于可以在使用 O 系列模型的 API 的过程中看到它们的思维链了！
        - 后台任务（Background Mode） ：这是最具变革性的功能之一。我们终于可以支持发起长时间运行的异步任务，不仅仅解决了复杂 Agent 任务中常见的网络超时问题，还使得用户可以在这个过程中关闭页面去干别的事情。
        - 远程 MCP Server 调用（Remote MCP Server）：直接支持了 MCP Server（Model Context Protocol）的调用，允许动态地调用外部服务器、甚至开发者自己部署的工具集，为 Agent 的能力扩展提供了无限可能。
- https://platform.openai.com/docs/guides/latest-model
    - The biggest difference, and main reason to migrate from Chat Completions to the Responses API for GPT-5, is support for **passing chain of thought (CoT)** between turns. See a full comparison of the APIs.
- https://lobehub.com/zh/blog/openai-responses-api-intergration

In [1]:
from openai import OpenAI
from dotenv import load_dotenv

assert load_dotenv()

In [2]:
client = OpenAI()

In [5]:
resp = client.responses.create(
    model='gpt-4o-mini',
    input='如何评价 Sam Altman'
)

In [7]:
print(resp.output_text)

Sam Altman 是一位在科技和创业界颇有影响力的人物。他曾是 Y Combinator 的总裁，Y Combinator 是全球最有名的创业孵化器之一，培养了许多成功的公司如 Dropbox 和 Airbnb。近年来，他因在人工智能领域的贡献而受到广泛关注，尤其是他在 OpenAI 的领导角色。

### 评价观点：

1. **创业与投资**：Altman 对科技创业的深刻理解和投资眼光，使他成为创业者和投资者共同信任的人物。

2. **推动人工智能**：在 OpenAI 的工作使他在 AI 领域有了显著的影响力，他强调负责任地开发和使用人工智能技术。

3. **思想领袖**：Altman 在公开场合和社交媒体上分享观点，常常引发对科技未来的讨论，他关注 AI 的伦理和社会影响，显示出对人文关怀的关注。

4. **争议与挑战**：由于他推动技术的快速发展，有人对技术可能带来的社会问题表示担忧，这也引发了对他及其工作的批评。

总的来说，Sam Altman 被视为具有前瞻性思维的领袖，同时也面临着与快速变化的科技带来的伦理和社会挑战。


In [9]:
resp = client.responses.create(
    model='gpt-4o-mini',
    instructions='以藏头诗的风格输出',
    input='如何评价 Sam Altman'
)

In [10]:
print(resp.output_text)

**山巅一座高峰立，  
Altman才华通四方。  
睿智引领风潮起，  
梦绘未来志难忘。**


### role

In [11]:
resp = client.responses.create(
    model='gpt-4o-mini',
    input=[
        {
            'role': 'developer',
            'content': '以十四行诗的风格输出'
        },
        {
            'role': 'user',
            'content': '如何评价 Sam Altman'
        }
    ]
)

In [12]:
print(resp.output_text)

Sam Altman，一位杰出的领军者，  
他在科技界独树一帜的光辉，  
投资与创新的双重赋予，  
推动者，思想者，前路永不黯然。

从Y Combinator的掌舵者，  
引领初创的浪潮，激扬创业之梦，  
而后执掌OpenAI，探索未来边界，  
人工智能的愿景，稳步而深邃。

他的眼光广阔，洞悉时代脉搏，  
慈善与科技，齐驱并进共融，  
面对挑战，胸怀开阔的理想，  
勇于担责，推动社会的新解。

不仅是企业家，更是思想的引导，  
在变革浪潮中，永葆探索的心。


### stream

Responses API 则采用了一种全新的 迭代器模式（Iterator Pattern） 。它的流式数据不再是 choices，而是一个描述 Agent 执行状态的 items 序列。每个 item 可以是 
- message （对话消息）、
- reasoning （思考过程）、
- tool_use （工具使用）、
- image_generation（图像生成） 等等。

In [15]:
resp = client.responses.create(
    model='gpt-4o-mini',
    input=[
        {
            'role': 'developer',
            'content': '以十四行诗的风格输出'
        },
        {
            'role': 'user',
            'content': '正反两方面评价 Sam Altman'
        }
    ],
    stream=True
)
for event in resp:
    if event.type == 'response.output_text.delta':
        print(event.delta, end='')

**正面评价：**

在技术的潮流中，Sam Altman如星辰般闪耀，  
推动创新，带领众人勇敢探索，  
他的愿景宏图，燃起无数创业者的梦想，  
以智慧为舵，驶向未来的海洋。  

孵化平台，点亮无数希望的火花，  
提携创始人，给予真知灼见的支持，  
智慧与勇气，他的团队如同家人，  
共同追寻，在科技中找寻意义之旅。

**反面评价：**

然而前路荆棘，总有阴影潜伏身旁，  
面临伦理，技术的边界需小心捍卫，  
在追逐成功的背后，若心太执迷狂，  
或许会失去，那本应珍视的温暖。  

决策之中，难免孕育争议的风波，  
滑落向商业，让初心模糊而远，  
在辉煌的表象下，深思胜于盲从，  
或埋怨，或期待，仍需理智对待前行。  

### multi-turn interaction

In [18]:
context = [
    { "role": "user", "content": "What is the capital of France?" }
]
res1 = client.responses.create(
    model="gpt-5-mini",
    input=context,
)

# Append the first response’s output to context
print(res1.output)
context += res1.output

# Add the next user message
context += [
    { "role": "user", "content": "And it's population?" }
]

res2 = client.responses.create(
    model="gpt-5-mini",
    input=context,
)
print(res2.output)

[ResponseReasoningItem(id='rs_0f48360fa92d727f0068e5d8db647c8191af64c61a2d33bec0', summary=[], type='reasoning', encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0f48360fa92d727f0068e5d8ddd57881919190ebbb4041d07f', content=[ResponseOutputText(annotations=[], text='The capital of France is Paris.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')]
[ResponseReasoningItem(id='rs_0f48360fa92d727f0068e5d8df00748191a1fbdff14b0363e4', summary=[], type='reasoning', encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0f48360fa92d727f0068e5d8ec340481918c8dd3e7c920d13e', content=[ResponseOutputText(annotations=[], text='Paris city proper has about 2.1 million residents (roughly 2.1–2.2 million in recent estimates). The larger Paris metropolitan area depends on the definition but contains roughly 10–12 million people (Île‑de‑France region ≈12.2 million). If you want a precise, up‑to‑date figure I can look up the latest INSE

In [19]:
res1 = client.responses.create(
    model="gpt-4o-mini",
    input="What is the capital of France?",
    store=True
)

res2 = client.responses.create(
    model="gpt-4o-mini",
    input="And its population?",
    previous_response_id=res1.id,
    store=True
)

In [20]:
res2.output_text

"As of 2023, the population of Paris is approximately 2.1 million people. However, the larger metropolitan area has a population of around 12 million. Population figures can vary, so it's a good idea to check the latest statistics for the most accurate numbers."

### reasoning model

- reasoning content

### langchain integration

- `use_responses_api = True`